# 03 — Grid Search Results & Model Selection
Run this notebook **after** `python src/train.py` has completed (or is still running — the Excel is updated after every run so you can inspect intermediate results).

Goals:
- Inspect all runs sorted by validation accuracy
- Identify the best configuration
- Understand which hyperparameters matter most (sensitivity analysis)
- Export enriched Excel with multiple sheets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = Path('../results')
EXCEL_PATH  = RESULTS_DIR / 'grid_search_results.xlsx'

df = pd.read_excel(EXCEL_PATH)
df_sorted = df.sort_values('final_val_acc', ascending=False).reset_index(drop=True)

print(f'Runs completed : {len(df)}')
print(f'Best val_acc   : {df["final_val_acc"].max():.4f}')
print(f'Best val_auc   : {df["final_val_auc"].max():.4f}')

## Top 10 configurations

In [ ]:
display(
    df_sorted[[
        'run_id', 'batch_size', 'unfreeze_from_layer',
        'phase1_lr', 'phase1_max_epochs',
        'phase2_lr', 'phase2_max_epochs',
        'p1_best_val_acc', 'p2_best_val_acc',
        'final_val_acc', 'final_val_auc', 'p2_overfit_gap'
    ]]
    .head(10)
    .style
    .background_gradient(subset=['final_val_acc', 'final_val_auc'], cmap='Greens')
    .background_gradient(subset=['p2_overfit_gap'], cmap='Reds')
    .format({
        'phase1_lr'     : '{:.0e}',
        'phase2_lr'     : '{:.0e}',
        'final_val_acc' : '{:.4f}',
        'final_val_auc' : '{:.4f}',
        'p1_best_val_acc': '{:.4f}',
        'p2_best_val_acc': '{:.4f}',
        'p2_overfit_gap' : '{:.4f}',
    })
)

## Sensitivity analysis
Mean and max validation accuracy grouped by each parameter value.
High spread across values = that parameter has a strong effect on outcome.

In [ ]:
params = [
    'batch_size', 'unfreeze_from_layer',
    'phase1_lr', 'phase1_max_epochs',
    'phase2_lr', 'phase2_max_epochs',
]

sensitivity_rows = []
for param in params:
    g = df.groupby(param)['final_val_acc'].agg(['mean', 'max', 'std']).reset_index()
    g.insert(0, 'parameter', param)
    g.columns = ['parameter', 'value', 'mean_val_acc', 'max_val_acc', 'std_val_acc']
    sensitivity_rows.append(g)

df_sensitivity = pd.concat(sensitivity_rows, ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, param in zip(axes, params):
    sub = df_sensitivity[df_sensitivity['parameter'] == param].copy()
    sub['value'] = sub['value'].astype(str)
    ax.bar(sub['value'], sub['mean_val_acc'], yerr=sub['std_val_acc'],
           color='steelblue', alpha=0.8, capsize=4, width=0.5)
    ax.set_title(param, fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Mean val accuracy')
    lo = df['final_val_acc'].min()
    ax.set_ylim(max(0, lo - 0.03), 1.0)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Hyperparameter sensitivity — mean validation accuracy', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sensitivity_plots.png', bbox_inches='tight', dpi=130)
plt.show()
print('Saved → results/sensitivity_plots.png')

## Export enriched Excel (3 sheets)

In [ ]:
with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    df_sorted.to_excel(writer,          sheet_name='all_results', index=False)
    df_sorted.head(10).to_excel(writer, sheet_name='top_10',      index=False)
    df_sensitivity.to_excel(writer,     sheet_name='sensitivity',  index=False)

print(f'Saved → {EXCEL_PATH.resolve()}')
print(f'  Sheet 1 — all_results  : {len(df_sorted)} runs')
print(f'  Sheet 2 — top_10')
print(f'  Sheet 3 — sensitivity')

## Best configuration — carry into 04_evaluate.ipynb

In [ ]:
best = df_sorted.iloc[0]

print('=' * 55)
print('  Best configuration')
print('=' * 55)
print(f'  Run ID               : {int(best.run_id):03d}')
print(f'  Batch size           : {int(best.batch_size)}')
print(f'  Unfreeze from layer  : {int(best.unfreeze_from_layer)}')
print(f'  Phase 1 LR           : {best.phase1_lr:.0e}')
print(f'  Phase 1 epochs run   : {int(best.p1_epochs_run)}')
print(f'  Phase 2 LR           : {best.phase2_lr:.0e}')
print(f'  Phase 2 epochs run   : {int(best.p2_epochs_run)}')
print('─' * 55)
print(f'  Final val accuracy   : {best.final_val_acc:.4f}')
print(f'  Final val AUC        : {best.final_val_auc:.4f}')
print(f'  Final val loss       : {best.final_val_loss:.4f}')
print(f'  Overfit gap (P2)     : {best.p2_overfit_gap:.4f}')
print('=' * 55)
print()

run_tag = (f'run{int(best.run_id):03d}_'
           f'bs{int(best.batch_size)}_'
           f'ul{int(best.unfreeze_from_layer)}_'
           f'p1lr{best.phase1_lr:.0e}_p1ep{int(best.phase1_max_epochs)}_'
           f'p2lr{best.phase2_lr:.0e}_p2ep{int(best.phase2_max_epochs)}')

print(f'  Best checkpoint filename:')
print(f'  models/checkpoints/{run_tag}_best.keras')